In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:
df=spark.table("workspace.bronze.dwh_cust_info")
df.display()


## **Trim the string **columns****

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType,StringType):
        df=df.withColumn(field.name,trim(col(field.name)))
df.display()

## Standardize marital status and Gender

In [0]:



df_stand=df.withColumn('cst_marital_status',when(upper(col('cst_marital_status'))=='M','Married')\
    .when(upper(col('cst_marital_status'))=='S','Single')\
    .otherwise('n/a'))\
    .withColumn('cst_gndr',when(upper(col('cst_gndr'))=='M','Male').when(upper(col('cst_gndr'))=='F','Female').otherwise('n/a'))

df.display()


## **Dropping Nulls**

In [0]:
df_null=df_stand.dropna(subset=['cst_id'])


In [0]:
df_clean=df_null.dropna(how='any')

## **Renaming columns**

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df_clean = df_clean.withColumnRenamed(old_name, new_name)

## **Writing table to Silver schema**

In [0]:
df_clean.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")